# 🧠 Advanced State: Input/Output Schemas and Runtime Configuration

## Learning Objectives
In this notebook, you will learn:
1. **Single-schema state** - why mixing input, output, and internal-only fields in one `TypedDict` leaks implementation details to the caller
2. **Separate input/output schemas** - using `input_schema`/`output_schema` on `StateGraph` so callers only see a clean input/output shape
3. **Private state fields** - fields (like a call counter) that live in the graph's internal state but never appear in what the caller sends or receives
4. **Runtime configuration** - injecting per-invocation settings via `RunnableConfig` (e.g., response language) without storing them in state

## Prerequisites
- Familiarity with `StateGraph`, nodes, and edges (see `01_Foundations`)
- An LLM configured via `helpers.get_experientiallabs_llm` (or swap in any other `helpers` factory)
- `EXPERIENTIALLABS_API_KEY` set in a `.env` file at the project root

---
## 📦 Part 0: Environment Setup

Load environment variables and initialize the LLM shared by every demo in this notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.config import RunnableConfig
from langgraph.graph import END, START, StateGraph

from helpers import get_experientiallabs_llm

# Load API keys from the project's .env file
load_dotenv()

# Initialize the LLM used by every demo in this notebook
model = get_experientiallabs_llm()

print(f"🤖 LLM initialized: {model.model_name}")

---
## 🗂️ Part 1: Input & Output State

By default, a `StateGraph` has exactly one schema: whatever you pass to `StateGraph(...)` is
also what the caller must send in and what they get back. That's fine for a small demo, but it
means any internal bookkeeping field (like a call counter) leaks straight into the public
input/output shape. This part first shows that problem, then fixes it by giving the graph
separate input, output, and private schemas.

### Key Concepts:
- **Single shared schema**: the same `TypedDict` serves as input, output, *and* internal state
- **`input_schema` / `output_schema`**: restrict what a caller can send in and what they get back,
  independent of the graph's full internal state

### 1.1 One Shared Schema (the Baseline)

`ChatMessages` mixes all three concerns in a single `TypedDict`: `question` (input),
`answer` (output), and `llm_calls` (an internal-only counter). Because there's only one
schema, the caller sees `llm_calls` too — even though it's pure bookkeeping.

In [ ]:
# ============================================================================
# STATE SCHEMA: ChatMessages (Single Shared Schema)
# ============================================================================
class ChatMessages(TypedDict):
    question: str
    answer: str
    llm_calls: int

### 1.2 The `call_model` Node

Answers the question with the LLM, and increments `llm_calls` each time the node runs. The
node mutates and returns the whole state dict directly rather than a partial update — both
are valid ways to update state in LangGraph.

In [ ]:
# ============================================================================
# NODE: call_model - Answer the Question and Track Call Count
# ============================================================================
def call_model(state: ChatMessages):
    question = state["question"]
    llm_calls = state.get("llm_calls", 0)
    state["llm_calls"] = llm_calls + 1
    print("LLM_CALLS:", state["llm_calls"])

    response = model.invoke(input=question)
    state["answer"] = response.content
    return state

### 1.3 Build and Run

A single-node graph: `START → agent → END`. Watch the returned state — it includes
`llm_calls` alongside `question` and `answer`, since with only one schema there's no way to
hide it from the caller.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Single Shared Schema
# ============================================================================
workflow = StateGraph(ChatMessages)

workflow.add_node("agent", call_model)
workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

graph = workflow.compile()
print("✅ Graph compiled (single shared schema).")

In [ ]:
# ============================================================================
# RUN: Notice `llm_calls` Leaking Into the Output
# ============================================================================
graph.invoke(input={"question": "Whats the highest mountain in the world?"})

### 1.4 Splitting Into Input, Private, and Output Schemas

Now we split the concerns into three focused schemas, and combine them into one `OverallState`
via multiple inheritance for the graph's *internal* view:
- **`InputState`**: exactly what the caller must provide (`question`)
- **`PrivateState`**: internal-only bookkeeping the caller never sees or sends (`llm_calls`)
- **`OutputState`**: exactly what the caller gets back (`answer`)
- **`OverallState`**: the union of all three — this is what nodes actually read and write

In [ ]:
# ============================================================================
# STATE SCHEMAS: Split Input / Private / Output
# ============================================================================
class InputState(TypedDict):
    question: str


class PrivateState(TypedDict):
    llm_calls: int


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, PrivateState, OutputState):
    pass

### 1.5 Build and Run With Separate Schemas

Passing `input_schema=InputState` and `output_schema=OutputState` alongside the full
`OverallState` restricts what the caller sends in and receives back — `llm_calls` stays purely
internal to the graph's execution.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Restrict Input/Output Shape
# ============================================================================
workflow = StateGraph(
    state_schema=OverallState,
    input_schema=InputState,
    output_schema=OutputState,
)

workflow.add_node("agent", call_model)
workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

graph = workflow.compile()
print("✅ Graph compiled (separate input/output schemas).")

In [ ]:
# ============================================================================
# RUN: `llm_calls` No Longer Appears in the Output
# ============================================================================
graph.invoke({"question": "Whats the highest mountain in the world?"})

---
## ⚙️ Part 2: Runtime Configuration

Sometimes a node needs per-invocation settings — like a response language — that shouldn't
live in state at all, since they're not part of the conversation data itself. LangGraph passes
a `RunnableConfig` into any node that declares a second parameter for it, and you read
per-call settings from `config["configurable"]`.

### Key Concepts:
- **`RunnableConfig`**: LangGraph injects this automatically when a node's signature accepts it
- **`config["configurable"]`**: a dict of arbitrary per-invocation settings you pass at `invoke()` time
- **Not state**: configuration values never appear in the graph's persisted state, unlike a regular field

### 2.1 A Config-Aware Node

This redefines `call_model` to accept a second `config: RunnableConfig` argument, reading a
`language` setting to control which language the LLM responds in. This overwrites Part 1's
`call_model` — from here on, that name refers to this version.

In [ ]:
# ============================================================================
# NODE: call_model - Config-Aware Version (Overwrites Part 1's Version)
# ============================================================================
def call_model(state: OverallState, config: RunnableConfig):
    language = config["configurable"].get("language", "English")
    system_message = SystemMessage(content=f"Respond in {language}")
    messages = [system_message, HumanMessage(content=state["question"])]

    response = model.invoke(messages)
    return {"answer": response.content}

### 2.2 Rebuild the Graph

Reusing the simpler `ChatMessages` schema from Part 1 for this standalone example — the point
here is the config injection, not the input/output schema split.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Reuse ChatMessages Schema
# ============================================================================
workflow = StateGraph(ChatMessages)

workflow.add_node("agent", call_model)
workflow.add_edge(START, "agent")
workflow.add_edge("agent", END)

graph = workflow.compile()
print("✅ Graph compiled (config-aware node).")

### 2.3 Run With Different Configs

Same question, same graph — only `config["configurable"]["language"]` changes between calls.
Note that `language` never appears in the returned state; it only ever lives in `config`.

In [ ]:
# ============================================================================
# RUN: Respond in Spanish
# ============================================================================
config = {"configurable": {"language": "Spanish"}}
graph.invoke({"question": "What's the highest mountain in the world?"}, config=config)

In [ ]:
# ============================================================================
# RUN: Respond in German
# ============================================================================
config = {"configurable": {"language": "German"}}
graph.invoke({"question": "What's the highest mountain in the world?"}, config=config)

---
## 📝 Summary

In this notebook, we learned:

### 1. Input & Output State
- **Single shared schema**: with only one `TypedDict`, every field — including internal
  bookkeeping like a call counter — leaks into the caller's input/output shape
- **`input_schema` / `output_schema`**: let a `StateGraph` expose a narrow input/output shape
  while nodes still operate on the full internal state
- **`PrivateState`**: a schema for fields that exist purely for the graph's own bookkeeping and
  never need to be sent in or returned

### 2. Runtime Configuration
- **`RunnableConfig`**: LangGraph automatically passes this into any node whose signature
  declares a second parameter for it
- **`config["configurable"]`**: the place to read per-invocation settings (language, user ID,
  feature flags, etc.) that shouldn't be persisted as state
- Configuration values are invisible in the returned state — they only affect *how* a node
  behaves during that one call

### Next Steps
- Continue to **[05_Subgraphs](../05_Subgraphs/)** to see how a graph's state is transformed
  when composed as a node inside a parent graph